In [ ]:
from google.colab import drive
drive.mount('/gdrive')

MessageError: Error: credential propagation was unsuccessful

# Check to see if we're running in Colab (versus local server)

In [ ]:
try:
  from google.colab import drive
  IN_COLAB=True
except:
  IN_COLAB=False

if IN_COLAB:
  print("We're running Colab")

# Tensorflow with GPU

This notebook provides an introduction to computing on a [GPU](https://cloud.google.com/gpu) in Colab. In this notebook you will connect to a GPU, and then run some basic TensorFlow operations on both the CPU and a GPU, observing the speedup provided by using the GPU.


## Enabling and testing the GPU

First, you'll need to enable GPUs for the notebook:

- Navigate to Edit→Notebook Settings
- select GPU from the Hardware Accelerator drop-down

Next, we'll confirm that we can connect to the GPU with tensorflow:

In [ ]:
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

## Observe TensorFlow speedup on GPU relative to CPU

This example constructs a typical convolutional neural network layer over a
random image and manually places the resulting ops on either the CPU or the GPU
to compare execution speed.

In [ ]:
import tensorflow as tf
import timeit

device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  print(
      '\n\nThis error most likely means that this notebook is not '
      'configured to use a GPU.  Change this in Notebook Settings via the '
      'command palette (cmd/ctrl-shift-P) or the Edit menu.\n\n')
  raise SystemError('GPU device not found')

def cpu():
  with tf.device('/cpu:0'):
    random_image_cpu = tf.random.normal((100, 100, 100, 3))
    net_cpu = tf.keras.layers.Conv2D(32, 7)(random_image_cpu)
    return tf.math.reduce_sum(net_cpu)

def gpu():
  with tf.device('/device:GPU:0'):
    random_image_gpu = tf.random.normal((100, 100, 100, 3))
    net_gpu = tf.keras.layers.Conv2D(32, 7)(random_image_gpu)
    return tf.math.reduce_sum(net_gpu)

# We run each op once to warm up; see: https://stackoverflow.com/a/45067900
cpu()
gpu()

# Run the op several times.
print('Time (s) to convolve 32x7x7x3 filter over random 100x100x100x3 images '
      '(batch x height x width x channel). Sum of ten runs.')
print('CPU (s):')
cpu_time = timeit.timeit('cpu()', number=10, setup="from __main__ import cpu")
print(cpu_time)
print('GPU (s):')
gpu_time = timeit.timeit('gpu()', number=10, setup="from __main__ import gpu")
print(gpu_time)
print('GPU speedup over CPU: {}x'.format(int(cpu_time/gpu_time)))

# Start of the code

In [ ]:
import os
# import tensorflow.compat.v1 as tf #import tensorflow as tf by CHS
import tensorflow as tf # by CHS
from tensorflow.keras.utils import to_categorical #from keras.utils import to_categorical by CHS
import glob
import cv2
import numpy as np
import time
import inspect

tf.compat.v1.disable_eager_execution() #by CHS
tf.compat.v1.disable_v2_behavior() #CHS

In [ ]:
ls

In [ ]:
def data_normalizer(data): # by CHS 20221122-24 to make all values into numbers between 0 and 1: from 0 to 255 into between 0 and 1 by dividing 255 to all values
  import numpy as np

  data_normalized = data/(np.max(data)-np.min(data)) # 255 - 0 = 255
  return data_normalized

# **Dataset: ESC50 2024-09-11 Start**

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
# VGG_MEAN = [103.939, 116.779, 123.68]


def load_audio_data(data_normal_flag=True): # by CHS #data_normal_flag=True by CHS 20221122
    #dirs = ['Air Conditioner', 'Car Horn', 'Children Playing', 'Dog Bark'] #21/11/2022 code run experiment
    #dirs = ['Car Horn', 'Children Playing', 'Dog Bark'] #21/11/2022 code run experiment
    # dirs = ['Air Conditioner', 'Car Horn', 'Children Playing', 'Dog Bark', 'Drilling', 'Engine Idling', 'Gun Shot', 'Jackhammer', 'Siren', 'Street Music']
    # [Hyosun] ESC dataset: 2024-09-11
    dirs = ['Dog', 'Rooster', 'Pig', 'Cow', 'Frog', 'Cat', 'Hen', 'Insects(Flying)', 'Sheep', 'Crow',
            'Rain', 'Sea waves', 'Crackling fire', 'Crickets', 'Chirping birds', 'Water drops', 'Wind', 'Pouring water', 'Toilet flush', 'Thunderstorm',
            'Crying baby', 'Sneezing', 'Clapping', 'Breathing', 'Coughing', 'Footsteps', 'Laughing', 'Brushing teeth', 'Snoring', 'Drinking,sipping',
            'Door knock', 'Mouse click', 'Keyboard typing', 'Door, wood creaks', 'Can opening', 'Washing machine', 'Vacuum cleaner', 'Clock alarm', 'Clock tick', 'Glass breaking',
            'Helicopter', 'Chainsaw',' Siren', 'Car horn', 'Engine', 'Train', 'Church bells', 'Airplane', 'Fireworks', 'Hand saw']
    # [/Hyosun] ESC dataset: 2024-09-11
    data       = [] # by CHS
    lab = []

    import random #[Hyosun] add 2024-09-07
    i = 0 # The label index in the order above in dirs

    for i_dir_ in range(len(dirs)): #[Hyosun] 2024-09-11 #for dir_ in dirs:
        random.seed(306) #[Hyosun] add 2024-09-07
        # Python glob. glob() method returns a list of files or folders that matches the path specified in the pathname argument.
        # This function takes two arguments, namely pathname, and recursive flag.
        # pathname : Absolute (with full path and the file name) or relative (with UNIX shell-style wildcards)
        files = glob.glob(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/images', str(i_dir_),'*.jpg')) #[Hyosun]remove .DS_Store first -> fixing the error sorted out!

        files_cnt_each_label = 0
        for file in files:
            # if files_cnt_each_label < 10000: # [Hyosun] added if statement 2024-09-07
            data.append(cv2.imread(file, 1))
            files_cnt_each_label += 1
            # else:   # [Hyosun] added if statement 2024-09-07
            #     break
        # print("files_cnt_each_label: ", files_cnt_each_label) #[Hyosun] added 2024-09-07
        lab = np.append(lab, np.tile(i, files_cnt_each_label), axis = 0)
        # print("lab: ", lab)                                   #[Hyosun] added 2024-09-07
        i +=1
    labels = to_categorical(lab) # one-hot conversion # train, test: each 200 개씩 있음
    # data normalization by CHS 20221122 -------
    if data_normal_flag == True:
      data_normalized = data_normalizer(np.array(data))
    # ------------------------------------------
    # print("data_normalized:\n", np.array(data_normalized)[:10], "\n labels: \n", labels[:10]) #[Hyosun] added 2024-09-07
    return np.array(data_normalized), labels # by CHS data-> data_normalized 20221122

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
# VGG_MEAN = [103.939, 116.779, 123.68]

# This is also the right version of function confirmed by CHS 20221202
def load_audio_data_v2(test_fraction=0.2, data_normal_flag=True):
              #[Hyosun]test_fraction=0.2 modified 2024-09-07
    """ by CHS : data_normal_flag=True by CHS 20221122
               : test_fraction=0.3     by CHS 20221124
    """

    #dirs = ['Air Conditioner', 'Car Horn', 'Children Playing', 'Dog Bark'] #21/11/2022 code run experiment
    #dirs = ['Car Horn', 'Children Playing', 'Dog Bark'] #21/11/2022 code run experiment
    # [Hyosun] esc dataset: 2024-09-11
    dirs = ['Dog', 'Rooster', 'Pig', 'Cow', 'Frog', 'Cat', 'Hen', 'Insects(Flying)', 'Sheep', 'Crow',
            'Rain', 'Sea waves', 'Crackling fire', 'Crickets', 'Chirping birds', 'Water drops', 'Wind', 'Pouring water', 'Toilet flush', 'Thunderstorm',
            'Crying baby', 'Sneezing', 'Clapping', 'Breathing', 'Coughing', 'Footsteps', 'Laughing', 'Brushing teeth', 'Snoring', 'Drinking,sipping',
            'Door knock', 'Mouse click', 'Keyboard typing', 'Door, wood creaks', 'Can opening', 'Washing machine', 'Vacuum cleaner', 'Clock alarm', 'Clock tick', 'Glass breaking',
            'Helicopter', 'Chainsaw',' Siren', 'Car horn', 'Engine', 'Train', 'Church bells', 'Airplane', 'Fireworks', 'Hand saw']
    # [/Hyosun] esc dataset: 2024-09-11
    data       = [] # by CHS
    data_train = []
    data_test  = []
    lab        = [] # by CHS : doesn't have to be numpy
    lab_train  = [] # by CHS
    lab_test   = [] # by CHS

    import random #[Hyosun] add 2024-09-07
    i = 0 # The label index in the order above in dirs
    for dir_ in dirs:
        random.seed(306) #[Hyosun] add 2024-09-07
        # Python glob. glob() method returns a list of files or folders that matches the path specified in the pathname argument.
        # This function takes two arguments, namely pathname, and recursive flag.
        # pathname : Absolute (with full path and the file name) or relative (with UNIX shell-style wildcards)
        files = glob.glob(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/images', dir_,'*.jpg'))

        files_cnt_each_label = 0
        data_each_label = [] # by CHS
        lab_each_label  = [] # by CHS

        # make data
        for file in files:
            if files_cnt_each_label < 10000: # [Hyosun] added if statement 2024-09-07
                data_each_label.append(cv2.imread(file, 1))
                files_cnt_each_label += 1
            else:   # [Hyosun] added if statement 2024-09-07
                break
        # make labels
        lab_each_label = np.tile(i, files_cnt_each_label)
        # lab_each_label = np.append(lab_each_label, np.tile(i, files_cnt_each_label), axis = 0)

        # split data(of each label) into train_set and test_set by CHS 20221124 ------------------------
        test_data_size_each_label  = int(files_cnt_each_label*test_fraction)
        train_data_size_each_label = files_cnt_each_label - test_data_size_each_label

        data_train.extend(data_each_label[:train_data_size_each_label])
        data_test.extend(data_each_label[train_data_size_each_label:])
        lab_train.extend(lab_each_label[:train_data_size_each_label])
        lab_test.extend(lab_each_label[train_data_size_each_label:])
        # -------------------------------------------------------------------------------

        i +=1

    labels_train = to_categorical(lab_train) # one-hot conversion
    labels_test  = to_categorical(lab_test)  # one-hot conversion

    # data normalization by CHS 20221122 -------
    if data_normal_flag == True:
      data_train_normalized = data_normalizer(np.array(data_train))
      data_test_normalized  = data_normalizer(np.array(data_test))
    # ------------------------------------------
    return np.array(data_train_normalized), labels_train, np.array(data_test_normalized), labels_test  # by CHS data-> data_normalized 20221122

In [ ]:
# a = [1, 2, 3]
# # a = 1
# b = []
# b.extend(a)
# print(b)
# b.extend(a)
# print(b)

In [ ]:
# train_data, train_labels, test_data, test_labels = load_audio_data_v2()
# train_data_len = len(train_data)
# test_data_len  = len(test_data)

In [ ]:
data, labels = load_audio_data()
# train_data_len = len(train_data)
# test_data_len  = len(test_data)

In [ ]:
# print(len(train_data))
# print(len(test_data))
# print(len(train_data)+len(test_data))

In [ ]:
# train_labels[-10:]

In [ ]:
# print(np.min(train_data))
# print(np.max(train_data))
# print(np.min(test_data))
# print(np.max(test_data))

In [ ]:
# import matplotlib.pyplot as plt
# plt.imshow(data[0])

In [ ]:
class Vgg19(tf.keras.Model): #[Hyosun] tf.keras.Model added 2024-09-08
    def __init__(self, data_len=400, no_labels=10, vgg19_npy_path=None, data_name="", fusion_type=""): # data_len=400, no_labels=10, fusion_type added(2024-09-03) data_name="" (2024-09-11) by CHS
        super().__init__() #[Hyosun] tf.keras.Model added 2024-09-08
        try:
            # print("ls: ", ls)
            self.data_dict = np.load('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/vgg19.npy', allow_pickle=True, encoding='latin1').item() # , allow_pickle=True by CHS
            # The default allow_pickle was changed for safety reasons.
            # It reduces the chances of loading something bad from the file.
            # You have to explicitly allow a pickle load for a file from a trusted source.
            # Without pickle, the load can only be a safe, numeric (or string) array. – hpaulj  May 4, 2020 at 0:41
        except:
            print("Place vgg19.npy in the same directory as of this notebook.")
            print("You can get it from ")
        # self.var_dict = {} #[Hyosun] added 2024-09-08
        self.isTrain = True
        self.epochs = 60 #80 #120 #80 #1 #60 #50 #1 #50 #the original epochs
        self.batch_size = 10
        self.data_size = data_len #400
        self.restore = False
        self.decay = 0.99
        self.model_name = 'model'
        self.ckpt_dir = '/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/' + self.model_name + "/checkpoint"
                                #'./' + self.model_name + "/checkpoint" #'/gdrive/MyDrive/ColabNotebooks/VGG19-Transfer-Learn-master/' by CHS
        self.test_fraction = 0.2 #[Hyosun] 2024-09-07 #0.5 # use the half of the train_set aas test_set (by CHS)
        self.no_labels = no_labels #2 #4 #10 # no_labels by CHS 21/11/2022
        # [Hyosun] for fusion added 2024-08-31
        self.layers_list = ["conv1_1", "conv1_1_relu", "conv1_2", "conv1_2_relu","pool1",
                            "conv2_1", "conv2_1_relu", "conv2_2", "conv2_2_relu", "pool2",
                            "conv3_1", "conv3_1_relu", "conv3_2", "conv3_2_relu", "conv3_3", "conv3_3_relu", "conv3_4", "conv3_4_relu", "pool3",
                            "conv4_1", "conv4_1_relu", "conv4_2", "conv4_2_relu", "conv4_3", "conv4_3_relu", "conv4_4", "conv4_4_relu", "pool4",
                            "conv5_1", "conv5_1_relu", "conv5_2", "conv5_2_relu", "conv5_3", "conv5_3_relu", "conv5_4", "conv5_4_relu", "pool5",
                            "fc6", "relu6", "fc7", "relu7", "fc8"]
        self.stride = 32
        # [/Hyosun] for fusion added 2024-08-31
        self.data_name = data_name     #[Hyosun]data_name added 2024-09-11
        self.fusion_type = fusion_type #[Hyosun]fusion_type added 2024-09-03
    def model(self):
        """
        load variable from npy to build the VGG

        :param rgb: rgb image [batch, height, width, 3] values scaled [0, 1]
        """

        with tf.compat.v1.variable_scope('Input') as scope:   #.compat.v1 by CHS
            self.rgb = tf.compat.v1.placeholder(name = "Input", shape = [None, 224, 224, 3], dtype = tf.float32)  #[None, 224, 224, 3], dtype = tf.float32) by CHS #.compat.v1 by CHS
            self.labels = tf.compat.v1.placeholder(name = "Labels", shape = [None, self.no_labels], dtype = tf.float32) #4], dtype = tf.float32) #.compat.v1 by CHS
            self.isTrain = tf.compat.v1.placeholder(name = "isTrain", shape = None, dtype = tf.bool) #.compat.v1 by CHS
            self.step = tf.compat.v1.train.get_or_create_global_step() #tf.train.get_or_create_global_step() #.compat.v1 by CHS
            self.lr = tf.compat.v1.placeholder(name = "LR", shape = None, dtype = tf.float32) #.compat.v1 by CHS

        # Convert RGB to BGR
        red, green, blue = tf.split(axis=3, num_or_size_splits=3, value=self.rgb)
        bgr = tf.concat(axis=3, values=[
            blue,  # - VGG_MEAN[0], by CHS
            green, # - VGG_MEAN[1], by CHS
            red,   # - VGG_MEAN[2], by CHS
        ])

        # [Hyosun: comment] Conv block layers Start 2024-08-30 ================================
        # :param rgb: rgb image [batch, height=224, width=224, 3] values scaled [0, 1]
        fusion_layers=[17, 35, 40] #initial test, needs to find the best combination
        maxpool_layers = [4, 9, 18, 27, 36] #[Hyosun] should be modified for vgg19
        # outputs = [layer.output for layer in self.layers]
        # self.layers_list = [layer for layer in self.layers]
        print("self.layers_list: ", self.layers_list)
        self.features, self.stride = [], 32 #나는 32부터 시작
          # for l, m in enumerate(self.features.children()): #nn.Sequential안의 각 layer 별 내용들 m에 있다.
          #     x = m(x) # 각 layer 통과
          #     if l in maxpool_layers:
          #         stride = stride // 2
          #     if l in fusion_layers:  #(xT, xF)
          #         x_cur = F.max_pool2d(x, kernel_size=(stride, 1), stride=(stride, 1)) if stride > 1 else x.clone()
          #         B, xC, xT, xF = x_cur.shape
          #         x_cur = x_cur.transpose(2, 3).reshape(B, xC*xF, xT)
          #         features.append(x_cur)
        # [Hyosun: comment] Conv block layers End 2024-08-30

        self.conv1_1 = self.conv_layer(bgr, "conv1_1")
        self.conv1_2 = self.conv_layer(self.conv1_1, "conv1_2")
        self.pool1 = self.max_pool(self.conv1_2, 'pool1')


        # [Hyosun] testing 2024-08-30
        print("conv result, self.pool1.get_shape().as_list(): ", self.pool1.get_shape().as_list())  #, %d, %d, %d" % (B, xC, xT, xF))
        pool1_shape = self.pool1.get_shape().as_list()
        dim = 1
        for d in pool1_shape[1:]:
            dim *= d
        x = tf.reshape(self.pool1, [-1, dim])
        print("reshaped self.pool1: ", x.get_shape().as_list())
        # [/Hyosun] testing 2024-08-30

        self.conv2_1 = self.conv_layer(self.pool1, "conv2_1")
        self.conv2_2 = self.conv_layer(self.conv2_1, "conv2_2")
        self.pool2 = self.max_pool(self.conv2_2, 'pool2')


        # [Hyosun] testing 2024-08-30
        print("conv result, self.pool2.get_shape().as_list(): ", self.pool2.get_shape().as_list())  #, %d, %d, %d" % (B, xC, xT, xF))
        pool2_shape = self.pool2.get_shape().as_list()
        dim = 1
        for d in pool2_shape[1:]:
            dim *= d
        x = tf.reshape(self.pool2, [-1, dim])
        print("reshaped self.pool2: ", x.get_shape().as_list())
        # [/Hyosun] testing 2024-08-30

        self.conv3_1 = self.conv_layer(self.pool2, "conv3_1")
        self.conv3_2 = self.conv_layer(self.conv3_1, "conv3_2")
        self.conv3_3 = self.conv_layer(self.conv3_2, "conv3_3")
        self.conv3_4 = self.conv_layer(self.conv3_3, "conv3_4")
        self.pool3 = self.max_pool(self.conv3_4, 'pool3')


        # [Hyosun] testing 2024-08-30
        print("conv result, self.pool3.get_shape().as_list(): ", self.pool3.get_shape().as_list())  #, %d, %d, %d" % (B, xC, xT, xF))
        pool3_shape = self.pool3.get_shape().as_list()
        dim = 1
        for d in pool3_shape[1:]:
            dim *= d
        x = tf.reshape(self.pool3, [-1, dim])
        print("reshaped self.pool3: ", x.get_shape().as_list())
        # [/Hyosun] testing 2024-08-30

        self.conv4_1 = self.conv_layer(self.pool3, "conv4_1")
        self.conv4_2 = self.conv_layer(self.conv4_1, "conv4_2")
        self.conv4_3 = self.conv_layer(self.conv4_2, "conv4_3")
        self.conv4_4 = self.conv_layer(self.conv4_3, "conv4_4")
        self.pool4 = self.max_pool(self.conv4_4, 'pool4')

        output = self.pool4 # 이런식으로 저장해놓으면 되겠네.. 헐 쉽네..
        print("[Hyosun] output: ", output)
        # [Hyosun] testing 2024-08-30
        print("conv result, self.pool4.get_shape().as_list(): ", self.pool4.get_shape().as_list())  #, %d, %d, %d" % (B, xC, xT, xF))
        pool4_shape = self.pool4.get_shape().as_list()
        dim = 1
        for d in pool4_shape[1:]:
            dim *= d
        x = tf.reshape(self.pool4, [-1, dim])
        print("reshaped self.pool4: ", x.get_shape().as_list())
        # [/Hyosun] testing 2024-08-30

        self.conv5_1 = self.conv_layer(self.pool4, "conv5_1")
        self.conv5_2 = self.conv_layer(self.conv5_1, "conv5_2")
        self.conv5_3 = self.conv_layer(self.conv5_2, "conv5_3")
        self.conv5_4 = self.conv_layer(self.conv5_3, "conv5_4")
        self.pool5 = self.max_pool(self.conv5_4, 'pool5')

        # [Hyosun] testing 2024-08-30 여기 shape이 그 shape 이 아닌데..난 x의 shape이 필요함
        print("conv result, self.pool5.get_shape().as_list(): ", self.pool5.get_shape())  #, %d, %d, %d" % (B, xC, xT, xF))
        pool5_shape = self.pool5.get_shape().as_list()
        dim = 1
        for d in pool5_shape[1:]:
            dim *= d
        x = tf.reshape(self.pool5, [-1, dim])
        print("reshaped self.pool5: ", x.get_shape().as_list())
        # [/Hyosun] testing 2024-08-30

        # [Hyosun: comment] FC block Start 2024-08-30 ============================================
        # B, xC, xT, xF = self.pool5.shape #pytorch syntax
        # self.pool5.get_shape().as_list()

          # unsqueeze: torch.unsqueeze(input, dim) → Tensor
          #            :Returns a new tensor with a dimension of size one inserted at the specified position
          # fc1_1, relu1_1, fc1_2, relu1_2, fc2, relu2 = list(self.fc.children())
          # x = x.permute(0, 2, 3, 1).contiguous()
          # x = x.view(x.size(0), -1)
          # x = fc1_1(x)
          # x = relu1_1(x)
          # if 17 in layers:
          #     x_cur = x.unsqueeze(-1).expand(-1, -1, 6)
                        # unsqueeze 2차 dim으로 늘리고, #expand: -1 argument는 해당 dim에 변화주지 않는다는 의미. 즉 마지막 dim에만 사이즈 6로 값을 반복해서 늘리겠다는 의미
          #     features.append(x_cur)
          # x = fc1_2(x)
          # x = relu1_2(x)
          # if 19 in layers:
          #     x_cur = x.unsqueeze(-1).expand(-1, -1, 6)
          #     features.append(x_cur)
          # x = fc2(x)
          # x = relu2(x)
          # if 21 in layers:
          #     x_cur = x.unsqueeze(-1).expand(-1, -1, 6)
          #     features.append(x_cur)
          # # fusion -> [B, *, 6]
          # x = torch.hstack(features)
          # return x
        # [Hyosun: comment] FC block End 2024-08-30
        self.fc6 = self.fc_layer(self.pool5, "fc6")
        assert self.fc6.get_shape().as_list()[1:] == [4096]
        # [Hyosun] testing
        print("[Hyosun] self.fc6 shape: ", np.shape(self.fc6))
        print("[Hyosun] self.fc6.get_shape().as_list(): ", self.fc6.get_shape().as_list())
        print("[Hyosun] self.fc6 type: ",  type(self.fc6))
        #[/Hyosun] testing
        self.relu6 = tf.nn.relu(self.fc6)

        # Hyosun 2024-08-28 added Start : hstack logic - one by one test 하면서 하기
        # Hyosun testing 2024-08-30:
        # self.mlp_head = tf.keras.ops.hstack(self.relu6)
        # self.mlp_head = np.hstack(self.relu6)
        print("[Hyosun] self.relu6 shape: ", np.shape(self.relu6))
        print("[Hyosun] self.relu6.get_shape().as_list(): ", self.relu6.get_shape().as_list())
        print("[Hyosun] self.relu6 type: ",  type(self.relu6))
        # x_cur = self.relu6.unsqueeze(-1).
        # tf.expand_dims(self.relu6, -1, -1, 6)
        # tf.concat([t1, t2], 1) : 결론, 요렇게 해야 맞을듯
        self.mlp_head = tf.concat([self.relu6, self.relu6], 1) #dimension reshape이 필요한듯
        print("[Hyosun] self.mlp_head shape: ", np.shape(self.mlp_head))
        print("[Hyosun] self.mlp_head type: ",  type(self.mlp_head))
        self.mlp_head = tf.concat([self.relu6, self.relu6], 1) #dimension reshape이 필요한듯
        print("[Hyosun] self.mlp_head shape: ", np.shape(self.mlp_head))
        print("[Hyosun] self.mlp_head type: ",  type(self.mlp_head))
        # Hyosun 2024-08-28 added End

        self.fc7 = self.fc_layer(self.relu6, "fc7")
        self.relu7 = tf.nn.relu(self.fc7)

        # [Hyosun] for fusion 2024-08-31
        # self.layers_list = ["conv1_1", "conv1_1_relu", "conv1_2", "conv1_2_relu","pool1",
        #                     "conv2_1", "conv2_1_relu", "conv2_2", "conv2_2_relu", "pool2",
        #                     "conv3_1", "conv3_1_relu", "conv3_2", "conv3_2_relu", "conv3_3", "conv3_3_relu", "conv3_4", "conv3_4_relu", "pool3",
        #                     "conv4_1", "conv4_1_relu", "conv4_2", "conv4_2_relu", "conv4_3", "conv4_3_relu", "conv4_4", "conv4_4_relu", "pool4",
        #                     "conv5_1", "conv5_1_relu", "conv5_2", "conv5_2_relu", "conv5_3", "conv5_3_relu", "conv5_4", "conv5_4_relu", "pool5",
        #                     "fc6", "relu6", "fc7", "relu7", "fc8"]
        print("[Hyosun] len(self.layers_list): ", len(self.layers_list))
        for i in range(len(self.layers_list)): # should be 41
            if i in [4, 9, 18, 27, 36]:
                self.stride = self.stride // 2
                print("self.stride: ", self.stride)
            if i in fusion_layers: #fusion_layers=[17, 35, 40]
                print("self.layers_list[%d]: %s"  %(i, self.layers_list[i]))
                if self.layers_list[i] == "conv1_2_relu":
                    xcur = self.conv1_2
                elif self.layers_list[i] == "conv2_2_relu":
                    xcur = self.conv2_2
                elif self.layers_list[i] == "conv3_4_relu":
                    xcur = self.conv3_4
                elif self.layers_list[i] == "conv4_4_relu":
                    xcur = self.conv4_4
                elif self.layers_list[i] == "conv5_4_relu":
                    xcur = self.conv5_4

                if self.layers_list[i] in ["conv1_2_relu", "conv2_2_relu", "conv3_4_relu", "conv4_4_relu", "conv5_4_relu"]:
                    B, xF, xT, xC = xcur.get_shape().as_list()
                    print("[Hyosun:Before hmax_pool] B:%s xC:%d xT:%d xF:%d" %(B, xF, xT, xC ))
                    xcur_dash = self.hmax_pool(xcur, "hmax_pooled_"+self.layers_list[i])
                    B, xF, xT, xC = xcur_dash.get_shape().as_list()
                    print("[Hyosun: After hmax_pool] B:%s xC:%d xT:%d xF:%d" %(B, xF, xT, xC ))
                    # xcur_hat = xcur_dash
                    # xcur_dash = tf.transpose(xcur_dash, perm=[0,1,3,2])
                    # xcur_hat  = xcur_dash
                    xcur_hat = tf.reshape(tf.transpose(xcur_dash, perm=[0,1,3,2]),[-1, xC*xF, xT])
                    # B: -1 setup meaning that B is automatically fixed so that the total size of tf remains the same.
                    #[B, xC*xF, xT] #B, xF, xC, xT
                    if self.layers_list[i] == "conv5_4_relu": #[B, 7168, 7] -> should be [B, 7168*2=14336, 7]
                        xcur_hat = tf.concat([xcur_hat, xcur_hat], 1)
                # elif self.layers_list[i] == "relu6":
                #     xcur = self.relu6
                #     B, xFxC = xcur.get_shape().as_list()
                #     print("[Hyosun:Before Expand dims, xcur] B:%s xFxC:%d" %(B, xFxC))
                #     print("[Hyosun] xcur.shape[1]//2: ", xcur.shape[1]//2)
                #     xcur = tf.concat([xcur,xcur,xcur, xcur[:,0:xcur.shape[1]//2]], 1)
                #     B, xFxC = xcur.get_shape().as_list()
                #     print("[Hyosun: After repeat 3.5 for dim xFxC, xcur] B:%s xFxC:%d" %(B, xFxC))
                #     xcur = tf.expand_dims(xcur, axis=2)
                #     B, xFxC, xT = xcur.get_shape().as_list()
                #     print("[Hyosun: After Expand dims, xcur] B:%s xFxC:%d xT:%d" %(B, xFxC, xT))
                #     xcur_hat = tf.concat([xcur, xcur, xcur, xcur, xcur, xcur, xcur], 2)
                #     B, xFxC, xT = xcur_hat.get_shape().as_list()
                #     print("[Hyosun: After Repeat xT=7 times of dim xT, xcur] B:%s xFxC:%d xT:%d" %(B, xFxC, xT))
                elif self.layers_list[i] == "relu6" or "relu7":
                    if self.layers_list[i] == "relu6":
                        xcur = self.relu6
                    elif self.layers_list[i] == "relu7":
                        xcur = self.relu7
                    B, xFxC = xcur.get_shape().as_list()
                    print("[Hyosun:Before Expand dims, xcur] B:%s xFxC:%d" %(B, xFxC))
                    print("[Hyosun] xcur.shape[1]//2: ", xcur.shape[1]//2)
                    xcur = tf.concat([xcur,xcur,xcur, xcur[:,0:xcur.shape[1]//2]], 1)
                    B, xFxC = xcur.get_shape().as_list()
                    print("[Hyosun: After repeat 3.5 for dim xFxC, xcur] B:%s xFxC:%d" %(B, xFxC))
                    xcur = tf.expand_dims(xcur, axis=2)
                    B, xFxC, xT = xcur.get_shape().as_list()
                    print("[Hyosun: After Expand dims, xcur] B:%s xFxC:%d xT:%d" %(B, xFxC, xT))
                    xcur_hat = tf.concat([xcur, xcur, xcur, xcur, xcur, xcur, xcur], 2)
                    B, xFxC, xT = xcur_hat.get_shape().as_list()
                    print("[Hyosun: After Repeat xT=7 times of dim xT, xcur] B:%s xFxC:%d xT:%d" %(B, xFxC, xT))
                print("xcur_hat shape : ", xcur_hat.get_shape().as_list())
                B, xFxC, xT = xcur_hat.get_shape().as_list()
                print("[Hyosun:Right Before Append into features, xcur_hat] B:%s xFxC:%d xT:%d" %(B, xFxC, xT))
                self.features.append(xcur_hat)
        self.fusion_output = tf.concat(self.features, 1)
        B, xFxC, xT = self.fusion_output.get_shape().as_list()
        print("[Hyosun:After Concat of features, self.fusion_output] B:%s xFxC:%d xT:%d" %(B, xFxC, xT))
        # Pooling combinations apply Start 2024-09-01 ===
        #### Apply Equation(3) to get a single vector for the input
        # x = mean_max_pooling(x)
        self.max_fusion_output  = tf.reduce_max(self.fusion_output, axis=[2]) #, keep_dims=True)#(self.fusion_output, ksize=[1, 1, 7], strides=[1, 1, 7], padding='SAME', name="max_fusion_output")
        self.mean_fusion_output = tf.reduce_mean(self.fusion_output,axis=[2]) #, keep_dims=True) #(self.fusion_output, ksize=[1, 1, 7], strides=[1, 1, 7], padding='SAME', name="mean_fusion_output")
        self.mean_max_fusion_output = self.mean_fusion_output + self.max_fusion_output
        print("mean_max_fusion_output shape : ", self.mean_max_fusion_output.get_shape().as_list())
        B, xFxC = self.mean_max_fusion_output.get_shape().as_list()
        print("[Hyosun: mean_max_fusion_output shape] B:%s xFxC:%d" %(B, xFxC))

        #[Hyosun] max+max_fusion added 2024-09-03 17:30 pm Start
        self.max_max_fusion_output = self.max_fusion_output + self.max_fusion_output
        print("max_max_fusion_output shape : ", self.max_max_fusion_output.get_shape().as_list())
        B, xFxC = self.max_max_fusion_output.get_shape().as_list()
        print("[Hyosun: max_max_fusion_output shape] B:%s xFxC:%d" %(B, xFxC))
        #[/Hyosun] max+max_fusion added 2024-09-03 17:30 pm End

        #[Hyosun] 2024-09-03 Start
        if self.fusion_type == "mean":
            self.final_fusion_output = self.mean_fusion_output
        elif self.fusion_type == "max":
            self.final_fusion_output = self.max_fusion_output
        elif self.fusion_type == "mean+max":
            self.final_fusion_output = self.mean_max_fusion_output
        elif self.fusion_type == "max+max":
            self.final_fusion_output = self.max_max_fusion_output
        else: # The original line
            self.final_fusion_output = self.relu7
        #[/Hyosun]2024-09-03 End

        # def mean_max_pooling(frame_embeddings, dim=-1):
        #     assert len(frame_embeddings.shape) == 3 # Batch,Feature Dimension,Time
        #     (x1, _) = torch.max(frame_embeddings, dim=dim)
        #     x2 = torch.mean(frame_embeddings, dim=dim)
        #     x = x1 + x2
        #     return x
        # pooling combinations apply End   2024-09-01 ===
        # [/Hyosun] for fusion 2024-08-31

        # [Hyosun] for fusion modified 2024-09-01
        # self.logits = self.fc_layer_output(self.relu7, "fc8") # [Hyosun:comment, the original line]The last layer: we seem to train only this last layer :) 2023-02-21 analysed
        self.logits = self.fc_layer_output(self.final_fusion_output, "fc8") # [Hyosun] for fusion added 2024-09-01, final_fusion_output 2024-09-03
        #self.logits = self.fc_layer_output(self.relu6, "fc7")
        # [/Hyosun] for fusion modified 2024-09-01

        with tf.compat.v1.variable_scope("Loss") as scope: # .compat.v1 by CHS
            self.loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits = self.logits, labels = self.labels))

        with tf.compat.v1.variable_scope("Accuracy") as scope: # .compat.v1 by CHS

            self.logits_max_args = tf.argmax(self.logits, axis = 1)
            self.labels_max_args = tf.argmax(self.labels, axis = 1) # can retrieve the predictions with this (by CHS)
            self.equal = tf.reduce_sum(tf.cast(tf.equal(self.logits_max_args, self.labels_max_args), tf.float32))
            self.batch_acc = tf.divide(self.equal, tf.cast(tf.shape(self.logits)[0], tf.float32))

        with tf.compat.v1.variable_scope("Optimizer") as scope: #.compat.v1 by CHS

            update_ops = tf.compat.v1.get_collection(tf.compat.v1.GraphKeys.UPDATE_OPS) #.compat.v1 by CHS
            with tf.control_dependencies(update_ops):
                self.optimizer = tf.compat.v1.train.AdamOptimizer(self.lr).minimize(self.loss, self.step)
                #tf.train.AdamOptimizer(self.lr).minimize(self.loss, self.step) by CHS
            self.saver = tf.compat.v1.train.Saver(max_to_keep = 3) # .compat.v1 by CHS
        return self.logits
    def avg_pool(self, bottom, name):
        return tf.nn.avg_pool(bottom, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME', name=name)

    def max_pool(self, bottom, name):
        # # [Hyosun] for fusion added 2024-08-31
        # self.stride = self.stride // 2
        # # [/Hyosun] for fusion added 2024-08-31
        return tf.nn.max_pool(bottom, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME', name=name)
    #[Hyosun] for maxpool(z1, T0) feature append
    def hmax_pool(self, bottom, name):      # instead of ksize=[1, 2, self.stride, 1], strides=[1, 2, self.stride, 1]
        return tf.nn.max_pool(bottom, ksize=[1, 1, self.stride, 1], strides=[1, 1, self.stride, 1], padding='SAME', name=name)
    #[/Hyosun] for maxpool(z1, T0) feature append
    def conv_layer(self, bottom, name):
        with tf.compat.v1.variable_scope(name): #.compat.v1 by CHS
            filt = self.get_conv_filter(name) # [Hyosun]retrieved from the vgg19.npy file (by CHS)

            conv = tf.nn.conv2d(bottom, filt, [1, 1, 1, 1], padding='SAME') # and use the saved weights info from vgg19.npy (by CHS)

            conv_biases = self.get_bias(name) # [Hyosun]retrieved from the vgg19.npy file (by CHS)
            bias = tf.nn.bias_add(conv, conv_biases)

            relu = tf.nn.relu(bias)
            return relu

    def fc_layer(self, bottom, name):
        with tf.compat.v1.variable_scope(name): #.compat.v1 by CHS
            shape = bottom.get_shape().as_list()
            dim = 1
            for d in shape[1:]:
                dim *= d
            x = tf.reshape(bottom, [-1, dim])

            weights = self.get_fc_weight(name) # [Hyosun]retrieved from the vgg19.npy file (by CHS)
            biases = self.get_bias(name)       # [Hyosun]retrieved from the vgg19.npy file (by CHS)

            # Fully connected layer. Note that the '+' operation automatically
            # broadcasts the biases.
            fc = tf.nn.bias_add(tf.matmul(x, weights), biases)

            return fc

    def fc_layer_output(self, x, name):  # [Hyosun]The last layer: we seem to train only this last layer :) 2023-02-21 analysed
        with tf.compat.v1.variable_scope(name): #.compat.v1 by CHS
            shape = x.get_shape().as_list()
            xavier = tf.initializers.glorot_uniform() #tf.contrib.layers.xavier_initializer() by CHS
            # replaced "tf.contrib.layers.xavier_initializer()" with "tf.initializers.glorot_uniform()" by CHS: tf.contrib is deprecated in Tensorflow 2.x.
            weights = tf.compat.v1.get_variable("Weight", shape=[shape[1], self.no_labels], initializer = xavier, dtype = tf.float32)
            #.compat.v1 by CHS #changed 4 into self.no_labels 21/11/2022: the output shape needs to be changed by the number of labels
            biases = tf.compat.v1.get_variable("Bias", shape=[self.no_labels], initializer = xavier, dtype = tf.float32)
            #.compat.v1 by CHS #changed 4 into self.no_labels 21/11/2022: the output shape needs to be changed by the number of labels

            # Fully connected layer. Note that the '+' operation automatically
            # broadcasts the biases.
            fc = tf.nn.bias_add(tf.matmul(x, weights), biases)

            return fc

    def get_conv_filter(self, name): # retrieve from vgg19.npy file (by CHS)
        initializer = self.data_dict[name][0]
        return tf.compat.v1.get_variable("Filter", initializer = initializer, dtype = tf.float32) #.compat.v1 by CHS

    def get_bias(self, name): # retrieve from vgg19.npy file (by CHS)
        initializer = self.data_dict[name][1]
        return tf.compat.v1.get_variable("Bias", initializer = initializer, dtype = tf.float32) #.compat.v1 by CHS

    def get_fc_weight(self, name): # retrieve from vgg19.npy file (by CHS)
        initializer = self.data_dict[name][0] # retrieved weights from vgg19.npy (by CHS)
        return tf.compat.v1.get_variable("Weight", initializer = initializer, dtype = tf.float32) #.compat.v1 by CHS

    # #[Hyosun] added 2024-09-08 save weights into our .npy file
    # def get_var_conv(self, initial_value, name, idx, var_name):
    #     if self.data_dict is not None and name in self.data_dict:
    #         value = self.data_dict[name][idx]
    #     else:
    #         value = initial_value

    #     # if self.isTrain: #self.trainable:
    #     #     var = tf.Variable(value, name=var_name)
    #     # else:
    #     #     var = tf.constant(value, dtype=tf.float32, name=var_name)

    #     self.var_dict[(name, idx)] = value #var
    #     assert value.get_shape() == initial_value.get_shape() #var.get_shape() == initial_value.get_shape()
    #     return value #var

    # def get_var_fc(self, initial_value, name, idx, var_name):
    #     if self.data_dict is not None and name in self.data_dict and self.load_weight_fc is True:
    #         value = self.data_dict[name][idx]
    #     else:
    #         value = initial_value

    #     # if self.isTrain: #self.trainable:
    #     #     var = tf.Variable(value, name=var_name)
    #     # else:
    #     #     var = tf.constant(value, dtype=tf.float32, name=var_name)

    #     self.var_dict[(name, idx)] = value #var
    #     assert value.get_shape() == initial_value.get_shape() #var.get_shape() == initial_value.get_shape()
    #     return value #var
    #     # assert var.get_shape() == initial_value.get_shape()
    #     # return var

    # def save_npy(self, sess, npy_path="/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/Saved_models/str(date.today())/Hyosun_proposed_" + fusion_type + "_vgg19_.npy"):
    #     assert isinstance(sess, tf.Session)

    #     data_dict = {}

    #     for (name, idx), var in list(self.var_dict.items()):
    #         var_out = sess.run(var)
    #         if name not in data_dict:
    #             data_dict[name] = {}
    #         data_dict[name][idx] = var_out

    #     np.save(npy_path, data_dict)
    #     print("File saved", npy_path)
    #     return npy_path
    # # def get_var_count(self):
    # #     count = 0
    # #     for v in list(self.var_dict.values()):
    # #         count += reduce(lambda x, y: x * y, v.get_shape().as_list())
    # #     return count
    # # [/Hyosun] added 2024-09-08

    def train(self, shuffle_flag=False): # shuffle_flag=False by CHS 21/11/2022
        # replaced "tf.reset_default_graph()" with: by CHS
        from tensorflow.python.framework import ops
        ops.reset_default_graph() #tf.reset_default_graph() ---------------------------

        print('Loading Data')
        # train_data, train_labels, test_data, test_labels = load_audio_data() # load_audio_data() by CHS
        #[Hyosun] add 2024-09-08
        import random
        random.seed(306)
        #[/Hyosun] add 2024-09-08
        data, labels = load_audio_data() # load_audio_data() by CHS 20221130
        print('Data Loaded')
        print("data shape: ", np.shape(data)) #[Hyosun] added 2024-08-30

        print('Loading Model')
        self.model()
        print('Model Loaded')

        if not os.path.exists(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/', self.model_name)):
                            #'/gdrive/MyDrive/ColabNotebooks/VGG19-Transfer-Learn-master/', self.model_name):
                            # '/gdrive/MyDrive/ColabNotebooks/VGG19-Transfer-Learn-master/', added by CHS 20221125
            # os.mkdir('/gdrive/MyDrive/ColabNotebooks/VGG19-Transfer-Learn-master/',self.model_name)
                   # '/gdrive/MyDrive/ColabNotebooks/VGG19-Transfer-Learn-master/', added by CHS 20221125
            os.mkdir('./' + self.model_name ) #이거좀 다시 고치기, 작동 안하는거 같음 20221125 by CHS

        # edit by CHS 20221124, make it alive again 20221130 by CHS ---
        test_data_size = int(self.data_size * self.test_fraction)
        train_data_size = self.data_size - test_data_size
        # # commented out by CHS 20221130  ----------------------------
        # train_data_size = len(train_data)
        # test_data_size  = len(test_data)
        # -------------------------------------------------------------

        train_batches = train_data_size // self.batch_size #the floor division // rounds the result down to the nearest whole number (by CHS): int로 만들어, 남는거 버림
        test_batches = test_data_size // self.batch_size

        #data = ((data/255) * 2) - 1 # [0,255] --> [-1,1]

        with tf.compat.v1.Session() as self.sess: # .compat.v1 by CHS

            init_op = tf.compat.v1.global_variables_initializer() # .compat.v1 by CHS
            self.sess.run(init_op)
            parameter_count = tf.reduce_sum([tf.reduce_prod(tf.shape(v)) for v in tf.compat.v1.trainable_variables()]) # .compat.v1 by CHS

            print('Parameters:', parameter_count.eval())
            start = 0
            acc_best = -1
            lr = 0.00001 #0.00001 #[Hyosun] change 2024-09-12 #0.0001 #original

            # edited by CHS: as train_size and test_size are different ------

            # keep track of the original indices of the whole data(= train + test) by CHS 20221130
            original_indices  = np.arange(self.data_size)
            original_data     = data   # 20221205 by CHS
            original_labels   = labels # 20221205 by CHS

            # shuffle the whole data first and then split
            # - No shuffle as we need to compare with other models with the same datasets afterwards, recommended by professor Li Zhang 17/11/2022
            # - But we can still shuffle if we keep track of the original indices of the whole data(= train + test) (by CHS) 20221130
            if shuffle_flag == True:
              #random seed to be added here by CHS 20221201
              #[Hyosun] 2024-09-12
              import random
              random.seed(306)
              #[/Hyosun] 2024-09-12
              shuffled_ind  = np.random.permutation(self.data_size)
              permuted_indices   = original_indices[shuffled_ind]
              print(f"permuted_indices : {permuted_indices}") # by CHS
              # 20221130 added by CHS   ---------# [Hyosun] to be added 2024-09-12 data len limited: permuted indices[:10000] ==> tau load data에서 폴더별로 개수 정해서 가져오기
              data   = data[permuted_indices]
              labels = labels[permuted_indices]
              # ---------------------------------# [/Hyosun] to be added 2024-09-12 data len limited: permuted indices[:10000]

            # split
            #   - split data and labels by CHS : included again by CHS 20221130
            train_data   = np.copy(data[0:train_data_size])
            test_data    = np.copy(data[train_data_size:])
            train_labels = np.copy(labels[0:train_data_size])
            test_labels  = np.copy(labels[train_data_size:])
            #   - split indices added by CHS 20221130
            if shuffle_flag == True:
              train_indices  = permuted_indices[0:train_data_size]
              test_indices   = permuted_indices[train_data_size:]
            else:
              train_indices  = original_indices[0:train_data_size]
              test_indices   = original_indices[train_data_size:]

            # commented out by CHS 20221130 (no need to shuffle again as we already shuffled above)
            # # # No shuffle as we need to compare with other models with the same datasets afterwards, recommended by professor Li Zhang 17/11/2022
            # # # train_indices, test_indices: set up lists of indices by CHS
            # # # when shuffle_flag == True, shuffle again here, train_set and test_set, separately. by CHS
            # if shuffle_flag == True:
            #   train_indices  = np.copy(np.random.permutation(train_data_size))
            #   test_indices   = np.copy(np.random.permutation(test_data_size))
            # else:
            #   train_indices  = np.arange(train_data_size)
            #   test_indices   = np.arange(test_data_size)

            # # in case the_data_size == the_test_size: (by CHS) We won't use this 3 lines as we have different sizes of train and test datasets. Oct/2022
            # data_perm = np.random.permutation(train_data_size)
            # train_perm = data_perm
            # test_perm = np.copy(data_perm)                    -------------

            print('Training commences from epoch ', start)

            for i in range(start, self.epochs):

                count = 0
                count_test = 0
                avg_loss = 0

                # no need 20221201 by CHS
                # # # No shuffle as we need to compare with other models with the same datasets afterwards, recommended by professor Li Zhang 17/11/2022
                # # # shuffle one more time for train_data
                # if shuffle_flag == True:
                #   shuffle = np.random.permutation(train_data_size)
                #   train_indices = train_indices[shuffle] #train_perm = train_perm[shuffle] (by CHS)
                #   print(f"train_indices : {train_indices }") # by CHS

                #[Hyosun] commented out 2024-09-12 : moved to below than acc :)
                if i+1>=30 and i+1==50: #(i+1)%40==0: #i+1 == 50: #[Hyosun] change (lr 0.0001)  50 -> 30 2024-09-12 20: 0.68, 30: 72.75, 40: 77 (lr 0.00001) 40: 77.75 (the best so far)
                  #i+1>=40 and added
                    lr = lr/10
                #[/Hyosun]

                for j in range(train_batches): #train_batches: train_data에서 뒤에 남는 몇개 뺀 가득찬 batch들의 개수, int 개수에 대해 batch수 만큼 loop돌아

                    begin = time.time()
                    print("j: ", j) # by CHS
                    print(f"What is the size of train_data?: {np.size(train_data)}")
                    if j != train_batches-1 : # i.e.즉 마지막 batch가 아닐 경우
                        current_batch = np.arange(j*self.batch_size, (j+1)*self.batch_size) #[j*self.batch_size : (j+1)*self.batch_size] #train_perm[j*self.batch_size : (j+1)*self.batch_size] (by CHS)
                        print(f"current_batch: {current_batch}")# by CHS
                        # print(f"train_data[current_batch[0]]: {train_data[current_batch[0]]}")# by CHS
                        current_batch_real_indices_from_the_whole_dataset = train_indices[j*self.batch_size : (j+1)*self.batch_size]
                        print(f"current_batch_real_indices_from_the_whole_dataset: {current_batch_real_indices_from_the_whole_dataset}")# by CHS
                        x = train_data[current_batch]   # train_data size == 0 ==> the problem at the moment by CHS
                        y = train_labels[current_batch]
                        # current_batch = train_indices[j*self.batch_size : (j+1)*self.batch_size] #train_perm[j*self.batch_size : (j+1)*self.batch_size] (by CHS)
                        # print(f"current_batch: {current_batch}")# by CHS
                        # print(f"train data[current_batch[0]]: {data[current_batch[0]]}")# by CHS
                        # x = data[current_batch]   #from train_data   -> data   editied by CHS 20221201 # train_data size == 0 ==> the problem at the moment by CHS
                        # y = labels[current_batch] #from train_labels -> labels editied by CHS 20221201
                    else: # i.e.즉 마지막 batch일 경우: 정수로 떨어진 batch제외 그 이후 남는 떨거지들을 그냥 마지막 batch에 껴넣어서 돌린다. 그래도 상관은 없음 (by CHS)
                        current_batch = np.arange(j*self.batch_size, len(train_data)) #[j*self.batch_size: ] # filled in this line as it disappeared (by CHS) 20221129
                        x = train_data[current_batch]
                        y = train_labels[current_batch]
                        # current_batch = train_indices[j*self.batch_size: ] # filled in this line as it disappeared (by CHS) 20221129
                        # x = data[current_batch]   #from train_data   -> data   editied by CHS 20221201
                        # y = labels[current_batch] #from train_labels -> labels editied by CHS 20221201

                    feed_train = {self.rgb: x, self.labels: y, self.lr: lr} #self.rgb에 x넣고, self.labels에 y넣고, self.lr에 lr넣고 (by CHS)

                    _, eq, loss = self.sess.run([self.optimizer, self.equal, self.loss], feed_dict = feed_train) # 넣고 드디어 돌리는 부분
                    count += eq
                    avg_loss += loss

                    line = 'Batch: %d Batch Accuracy: %.4f Loss: %.4f Time/Batch: %.4f' %(j, eq/len(current_batch), float(loss), time.time() - begin)
                    print(line, end ='\r')

                # save predictions after training by CHS 20221129-1207 -----
                x_data = train_data
                y_data = train_labels
                feed_trainset = {self.rgb: x_data, self.labels: y_data}
                predictions_train = self.sess.run(self.labels_max_args, feed_dict = feed_trainset)
                print("Epoch: %d" %(i), " After training: Predictions of trainset: \n", predictions_train)

                # - save train_indices, test_indices, predictions, real labels by CHS 20221201 to keep track of dataset indices

                # results = f"Epoch: {i} After training: Predictions of trainset: \n{str(predictions_train)}\n"
                # print(results)
                import datetime
                from datetime import date
                from time import gmtime
                results = ""
                with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/',self.model_name, self.data_name, 'predictions_train_'+str(date.today())+self.fusion_type+'.txt'), 'a') as f :
                    for i_p in range(len(predictions_train)):
                      result = "Epoch:"+str(i)+",\ttrain_indices["+str(i_p)+"]: real index of the whole data "+str(train_indices[i_p])+",\tprediction:"+ str(predictions_train[i_p])+\
                               ",\ttrain_labels:"+str(np.argmax(train_labels[i_p]))+"\n"
                      # print(result)
                      results += result
                    f.write(results)
                    # for pred_train in predictions_train:
                    #   f.write(str(pred_train)+"\n")
                f.close()

                # ----------------------------------------------------------


                for j in range(test_batches):

                    print('======================Testing====================', end = '\r')
                    begin = time.time()

                    if j != test_batches-1 :
                        current_batch = np.arange(j*self.batch_size, (j+1)*self.batch_size) #[j*self.batch_size: (j+1)*self.batch_size] #test_perm[j*self.batch_size: (j+1)*self.batch_size] (by CHS)
                        x = test_data[current_batch]
                        y = test_labels[current_batch]
                        # current_batch = test_indices[j*self.batch_size: (j+1)*self.batch_size] #test_perm[j*self.batch_size: (j+1)*self.batch_size] (by CHS)
                        # x = data[current_batch]   #from test_data    -> data   editied by CHS 20221201
                        # y = labels[current_batch] #from test_labels  -> labels editied by CHS 20221201
                    else:
                        current_batch = np.arange(j*self.batch_size, len(test_data)) #test_perm[j*self.batch_size: ] (by CHS)
                        x = test_data[current_batch]
                        y = test_labels[current_batch]
                        # current_batch = test_indices[j*self.batch_size: ] #test_perm[j*self.batch_size: ] (by CHS)
                        # x = data[current_batch]   #from test_data    -> data   editied by CHS 20221201
                        # y = labels[current_batch] #from test_labels  -> labels editied by CHS 20221201

                    feed_test = {self.rgb: x, self.labels: y}
                    eq = self.sess.run(self.equal, feed_dict = feed_test)
                    #print(eq)
                    count_test += eq


                accuracy = count / train_data_size
                accuracy_test = count_test / test_data_size
                avg_loss = avg_loss / train_batches
                #[Hyosun] inserted here 2024-09-12
                # if i+1 == 50: #[Hyosun] inserted here: change(lr 0.0001) 50 -> 30 2024-09-12 20: 0.68, 30: 72.75, (lr 0.00001) 40: 77.75
                #     lr = lr/10
                #[/Hyosun] inserted here 2024-09-12

                line = "Epoch: %d Train Acc: %.6f Test Acc: %.6f Average Loss/Batch: %.6f" %(i,accuracy,accuracy_test,avg_loss)
                print(line)

                with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/',self.model_name, self.data_name, 'logs_'+str(date.today())+self.fusion_type+'.txt'), 'a') as f :
                                # path '/gdrive/MyDrive/ColabNotebooks/VGG19-Transfer-Learn-master/',added by CHS 20221124
                    line += '\n'
                    f.write(line)


                # save predictions after training by CHS 20221129-1207 -----
                x_data = test_data
                y_data = test_labels
                feed_testset = {self.rgb: x_data, self.labels: y_data}
                predictions_test = self.sess.run(self.labels_max_args, feed_dict = feed_testset)
                print("Epoch: %d" %(i), " After training: Predictions of testset: \n", predictions_test)

                # - save train_indices, test_indices, predictions, real labels by CHS 20221201 to keep track of dataset indices

                # results = f"Epoch: {i} After training: Predictions of trainset: \n{str(predictions_train)}\n"
                # print(results)

                from datetime import date
                from time import gmtime
                results = ""
                with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/',self.model_name, self.data_name, 'predictions_test_'+str(date.today())+self.fusion_type+'.txt'), 'a') as f :
                    for i_p in range(len(predictions_test)):
                      result = "Epoch:"+str(i)+",\ttest_indices["+str(i_p)+"]: real index of the whole data "+str(test_indices[i_p])+",\tprediction:"+ str(predictions_test[i_p])+\
                               ",\ttest_labels:"+str(np.argmax(test_labels[i_p]))+"\n"
                      # print(result)
                      results += result
                    f.write(results)
                    # for pred_train in predictions_train:
                    #   f.write(str(pred_train)+"\n")
                f.close()

                # ----------------------------------------------------------



                # [Hyosun] save model by CHS 2024-09-07 - 08 20221201 -----
                # Save the weights
                #tf.keras.Model.save_weights(filepath='/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/'+self.model_name+'_'+str(date.today())+'_'+self.fusion_type+'_my_checkpoint')
                # tf.keras.models.save_model(self.model, filepath='/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/'+ self.model_name +'_' + str(date.today()) + '_' + self.fusion_type + '_checkpoint.keras')
                #tf.compat.v1.train.Saver(filename='/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/'+ self.model_name +'_' + str(date.today()) + '_' + self.fusion_type + '_checkpoint.keras')
                # [/Hyosun] save model by CHS 2024-09-07 - 08

                # --------------------------------
    # #[Hyosun] added 2024-09-08
    # def predict(self, input, saved_model_path=None):
    #     self.base_path = '/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/'
    #     file_path = self.base_path + saved_model_path
    #     self.model()
    #     self.load_weights(file_path)

    # def evaluate(self, input_data, input_labels, saved_model_path=None):
    # #[/Hyosun] added 2024-09-08

In [ ]:
# data_len=10000
from datetime import date
data_name = "esc50"
base_dir = '/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/'+data_name+"/"

In [ ]:
fusion_type="mean+max"
model_file = 'proposed_vgg19_' + str(date.today()) + '_' + data_name + '_' + fusion_type
net = Vgg19(data_len=len(data), no_labels=50, data_name="esc50", fusion_type="mean+max") # data_len=len(data), no_labels=3, by CHS
# net = Vgg19(data_len=10000, no_labels=10, fusion_type="mean+max") # data_len=len(data), no_labels=3, by CHS
net.train(shuffle_flag=True)
tf.keras.models.save_model(net, filepath=base_dir + model_file + '.keras')#, save_traces=False)
net.save_weights(filepath=base_dir + model_file + '.weights.h5')#, save_traces=False))

In [ ]:
# from datetime import date
# fusion_type="mean+max"
# tf.keras.models.save_model(net, filepath='/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/proposed_vgg19_' + str(date.today()) + '_' + data_name + '_' + fusion_type + '.keras')#, save_traces=False)

In [ ]:
# net.save_weights(filepath='/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/proposed_vgg19_' + str(date.today()) + '_' + fusion_type + '.weights.h5')#, save_traces=False))

In [ ]:
# net = Vgg19(data_len=len(data), no_labels=10, fusion_type="mean+max")
# net.load_weights(filepath='/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/proposed_vgg19_2024-09-08_mean+max.keras')

In [ ]:
# net = Vgg19(data_len=len(data), no_labels=10, fusion_type="mean+max")
# net.load_weights(filepath='/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/checkpoints/proposed_vgg19_2024-09-08_mean+max.weights.h5')

In [ ]:
# net.train(shuffle_flag=True)

In [ ]:
# net.evaluate(data,labels)
# yhat= net.predict(data[0])
# net.model()

In [ ]:
# train_data_size = 10
# train_indices = np.arange(train_data_size)#[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# print(train_indices)
# shuffle = np.random.permutation(train_data_size)
# print(shuffle)
# train_indices = train_indices[shuffle]
# print(train_indices)

In [ ]:
import os
current_file = 'logs_2024-09-11mean+max.txt'
current_file_name, ext =current_file.split('.')
# Open and extract the lines of the data file
with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/', data_name, current_file), 'r') as f:
    lines = f.readlines()
f.close()

epoch_list, train_acc_list_mean_max, test_acc_list_mean_max, avg_loss_batch_list_mean_max = [], [], [], []
# loop through lines of the file
for line in lines:
    # remove newline and extract data from line
    e, epoch, tr, tra, train_acc, te, tea, test_acc, avg, loss_batch, avg_loss_batch = line.strip().split(' ')
    epoch_list.append(int(epoch))
    train_acc_list_mean_max.append(float(train_acc))
    test_acc_list_mean_max.append(float(test_acc))
    avg_loss_batch_list_mean_max.append(float(avg_loss_batch))
    print("epoch: %s, train_acc: %s, test_acc: %s, avg_loss_batch: %s" % (epoch, train_acc, test_acc, avg_loss_batch))
    # print

In [ ]:
print(train_acc_list_mean_max)

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# fig, axs = plt.subplots(3,1,figsize=(6, 16))
# # fig.suptitle('[Emotions] Average Precision with various number of layers at mlp_head') #, fontdict={'fontsize': 10}, loc='center', )


# #mlp_head with 2 layers
# axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list, 'o-', markersize=3, label='train accuracy')
# # axs[0].plot([i for i in range(1, results_mlp2_mean[:,2].size+1)],    results_mlp2_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_meanmax[:,2].size+1)], results_mlp2_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_max[:,2].size+1)],     results_mlp2_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_maxmax[:,2].size+1)],  results_mlp2_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[0].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[0].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[0].set_yticks(np.arange(0, np.max(results_mlp2_maxmax[:,0])+0.05, 0.001)) #, rotation=45)
# # axs[0].set_ylim(0.216, 0.234)
# axs[0].set_xlabel('Epoch')
# axs[0].set_ylabel('Train accuracy')
# axs[0].grid(True, linestyle='-')
# # axs[0].set_title('Train accuracy', fontdict={'fontsize': 10}, loc='center', )
# # fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

# #mlp_head with 4 layers
# axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list, 'o-', markersize=3, label='test accuracy')
# # axs[1].plot([i for i in range(1, results_mlp4_mean[:,2].size+1)],    results_mlp4_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_meanmax[:,2].size+1)], results_mlp4_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_max[:,2].size+1)],     results_mlp4_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_maxmax[:,2].size+1)],  results_mlp4_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[1].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[1].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[1].set_yticks(np.arange(0, np.max(results_mlp4_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# # axs[1].set_ylim(0.212, 0.234)
# axs[1].set_xlabel('Epoch')
# axs[1].set_ylabel('Test accuracy')
# axs[1].grid(True, linestyle='-')
# # axs[1].set_title('Test accuracy', fontdict={'fontsize': 10}, loc='center', )
# # fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

# #mlp_head with 6 layers
# axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list, 'o-', markersize=3, label='average loss/batch')
# # axs[2].plot([i for i in range(1, results_mlp6_mean[:,2].size+1)],    results_mlp6_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_meanmax[:,2].size+1)], results_mlp6_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_max[:,2].size+1)],     results_mlp6_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_maxmax[:,2].size+1)],  results_mlp6_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[2].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[2].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[2].set_yticks(np.arange(0, np.max(results_mlp6_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# # axs[2].set_ylim(0.216, 0.234)
# axs[2].set_xlabel('Epoch')
# axs[2].set_ylabel('Average Loss/Batch')
# axs[2].grid(True, linestyle='-')
# # axs[2].set_title('Average Loss/Batch', fontdict={'fontsize': 10}, loc='center', )


# fig.savefig('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/'+current_file_name+ 'VGG19_transfer_train_test_acc_loss.png',dpi=300)

# mean fusion

In [ ]:
fusion_type = "mean"
model_file = 'proposed_vgg19_' + str(date.today()) + '_' + data_name + '_' + fusion_type
net = Vgg19(data_len=len(data), no_labels=50, data_name="esc50", fusion_type="mean") # data_len=len(data), no_labels=3, by CHS
net.train(shuffle_flag=True)
tf.keras.models.save_model(net, filepath=base_dir + model_file + '.keras')#, save_traces=False)
net.save_weights(filepath=base_dir + model_file + '.weights.h5')#, save_traces=False))

In [ ]:
import os
current_file = 'logs_2024-09-11mean.txt'
current_file_name, ext =current_file.split('.')
# Open and extract the lines of the data file
with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/',  data_name, current_file), 'r') as f:
    lines = f.readlines()
f.close()

epoch_list, train_acc_list_mean, test_acc_list_mean, avg_loss_batch_list_mean = [], [], [], []
# loop through lines of the file
for line in lines:
    # remove newline and extract data from line
    e, epoch, tr, tra, train_acc, te, tea, test_acc, avg, loss_batch, avg_loss_batch = line.strip().split(' ')
    epoch_list.append(int(epoch))
    train_acc_list_mean.append(float(train_acc))
    test_acc_list_mean.append(float(test_acc))
    avg_loss_batch_list_mean.append(float(avg_loss_batch))
    print("epoch: %s, train_acc: %s, test_acc: %s, avg_loss_batch: %s" % (epoch, train_acc, test_acc, avg_loss_batch))
    # print

In [ ]:
print(train_acc_list_mean)

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# fig, axs = plt.subplots(3,1,figsize=(6, 16))
# # fig.suptitle('[Emotions] Average Precision with various number of layers at mlp_head') #, fontdict={'fontsize': 10}, loc='center', )


# #mlp_head with 2 layers
# axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list, 'o-', markersize=3, label='train accuracy')
# # axs[0].plot([i for i in range(1, results_mlp2_mean[:,2].size+1)],    results_mlp2_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_meanmax[:,2].size+1)], results_mlp2_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_max[:,2].size+1)],     results_mlp2_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_maxmax[:,2].size+1)],  results_mlp2_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[0].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# axs[0].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[0].set_yticks(np.arange(0, np.max(results_mlp2_maxmax[:,0])+0.05, 0.001)) #, rotation=45)
# # axs[0].set_ylim(0.216, 0.234)
# axs[0].set_xlabel('Epoch')
# axs[0].set_ylabel('Train accuracy')
# axs[0].grid(True, linestyle='-')
# # axs[0].set_title('Train accuracy', fontdict={'fontsize': 10}, loc='center', )
# # fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

# #mlp_head with 4 layers
# axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list, 'o-', markersize=3, label='test accuracy')
# # axs[1].plot([i for i in range(1, results_mlp4_mean[:,2].size+1)],    results_mlp4_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_meanmax[:,2].size+1)], results_mlp4_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_max[:,2].size+1)],     results_mlp4_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_maxmax[:,2].size+1)],  results_mlp4_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[1].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[1].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[1].set_yticks(np.arange(0, np.max(results_mlp4_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# # axs[1].set_ylim(0.212, 0.234)
# axs[1].set_xlabel('Epoch')
# axs[1].set_ylabel('Test accuracy')
# axs[1].grid(True, linestyle='-')
# # axs[1].set_title('Test accuracy', fontdict={'fontsize': 10}, loc='center', )
# # fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

# #mlp_head with 6 layers
# axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list, 'o-', markersize=3, label='average loss/batch')
# # axs[2].plot([i for i in range(1, results_mlp6_mean[:,2].size+1)],    results_mlp6_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_meanmax[:,2].size+1)], results_mlp6_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_max[:,2].size+1)],     results_mlp6_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_maxmax[:,2].size+1)],  results_mlp6_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[2].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[2].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[2].set_yticks(np.arange(0, np.max(results_mlp6_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# # axs[2].set_ylim(0.216, 0.234)
# axs[2].set_xlabel('Epoch')
# axs[2].set_ylabel('Average Loss/Batch')
# axs[2].grid(True, linestyle='-')
# # axs[2].set_title('Average Loss/Batch', fontdict={'fontsize': 10}, loc='center', )


# fig.savefig('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/'+current_file_name+ 'VGG19_transfer_train_test_acc_loss.png',dpi=300)

In [ ]:
fusion_type = "max+max"
model_file = 'proposed_vgg19_' + str(date.today()) + '_' + data_name + '_' + fusion_type
net = Vgg19(data_len=len(data), no_labels=50, data_name = "esc50",fusion_type="max+max") # data_len=len(data), no_labels=3, by CHS
net.train(shuffle_flag=True)
tf.keras.models.save_model(net, filepath=base_dir + model_file + '.keras')#, save_traces=False)
net.save_weights(filepath=base_dir + model_file + '.weights.h5')#, save_traces=False))

In [ ]:
import os
current_file = 'logs_2024-09-11max+max.txt'
current_file_name, ext =current_file.split('.')
# Open and extract the lines of the data file
with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/',  data_name, current_file), 'r') as f:
    lines = f.readlines()
f.close()

epoch_list, train_acc_list_max_max, test_acc_list_max_max, avg_loss_batch_list_max_max = [], [], [], []
# loop through lines of the file
for line in lines:
    # remove newline and extract data from line
    e, epoch, tr, tra, train_acc, te, tea, test_acc, avg, loss_batch, avg_loss_batch = line.strip().split(' ')
    epoch_list.append(int(epoch))
    train_acc_list_max_max.append(float(train_acc))
    test_acc_list_max_max.append(float(test_acc))
    avg_loss_batch_list_max_max.append(float(avg_loss_batch))
    print("epoch: %s, train_acc: %s, test_acc: %s, avg_loss_batch: %s" % (epoch, train_acc, test_acc, avg_loss_batch))
    # print

In [ ]:
print(train_acc_list_max_max)

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# fig, axs = plt.subplots(3,1,figsize=(6, 16))
# # fig.suptitle('[Emotions] Average Precision with various number of layers at mlp_head') #, fontdict={'fontsize': 10}, loc='center', )


# #mlp_head with 2 layers
# axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list, 'o-', markersize=3, label='train accuracy')
# # axs[0].plot([i for i in range(1, results_mlp2_mean[:,2].size+1)],    results_mlp2_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_meanmax[:,2].size+1)], results_mlp2_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_max[:,2].size+1)],     results_mlp2_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[0].plot([i for i in range(1, results_mlp2_maxmax[:,2].size+1)],  results_mlp2_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[0].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[0].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[0].set_yticks(np.arange(0, np.max(results_mlp2_maxmax[:,0])+0.05, 0.001)) #, rotation=45)
# # axs[0].set_ylim(0.216, 0.234)
# axs[0].set_xlabel('Epoch')
# axs[0].set_ylabel('Train accuracy')
# axs[0].grid(True, linestyle='-')
# # axs[0].set_title('Train accuracy', fontdict={'fontsize': 10}, loc='center', )
# # fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

# #mlp_head with 4 layers
# axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list, 'o-', markersize=3, label='test accuracy')
# # axs[1].plot([i for i in range(1, results_mlp4_mean[:,2].size+1)],    results_mlp4_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_meanmax[:,2].size+1)], results_mlp4_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_max[:,2].size+1)],     results_mlp4_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[1].plot([i for i in range(1, results_mlp4_maxmax[:,2].size+1)],  results_mlp4_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[1].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[1].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[1].set_yticks(np.arange(0, np.max(results_mlp4_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# # axs[1].set_ylim(0.212, 0.234)
# axs[1].set_xlabel('Epoch')
# axs[1].set_ylabel('Test accuracy')
# axs[1].grid(True, linestyle='-')
# # axs[1].set_title('Test accuracy', fontdict={'fontsize': 10}, loc='center', )
# # fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

# #mlp_head with 6 layers
# axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list, 'o-', markersize=3, label='average loss/batch')
# # axs[2].plot([i for i in range(1, results_mlp6_mean[:,2].size+1)],    results_mlp6_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_meanmax[:,2].size+1)], results_mlp6_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_max[:,2].size+1)],     results_mlp6_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# # axs[2].plot([i for i in range(1, results_mlp6_maxmax[:,2].size+1)],  results_mlp6_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# # axs[2].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# # plt.xticks(rotation=45)
# # axs[2].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# # axs[2].set_yticks(np.arange(0, np.max(results_mlp6_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# # axs[2].set_ylim(0.216, 0.234)
# axs[2].set_xlabel('Epoch')
# axs[2].set_ylabel('Average Loss/Batch')
# axs[2].grid(True, linestyle='-')
# # axs[2].set_title('Average Loss/Batch', fontdict={'fontsize': 10}, loc='center', )


# fig.savefig('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/'+current_file_name+ 'VGG19_transfer_train_test_acc_loss.png',dpi=300)

In [ ]:
fusion_type = "max"
model_file = 'proposed_vgg19_' + str(date.today()) + '_' + data_name + '_' + fusion_type
net = Vgg19(data_len=len(data), no_labels=50, data_name="esc50",fusion_type="max") # data_len=len(data), no_labels=3, by CHS
net.train(shuffle_flag=True)
tf.keras.models.save_model(net, filepath=base_dir + model_file + '.keras')#, save_traces=False)
net.save_weights(filepath=base_dir + model_file + '.weights.h5')#, save_traces=False))

In [ ]:
import os
current_file = 'logs_2024-09-11max.txt'
current_file_name, ext =current_file.split('.')
# Open and extract the lines of the data file
with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/',  data_name,  current_file), 'r') as f:
    lines = f.readlines()
f.close()

epoch_list, train_acc_list_max, test_acc_list_max, avg_loss_batch_list_max = [], [], [], []
# loop through lines of the file
for line in lines:
    # remove newline and extract data from line
    e, epoch, tr, tra, train_acc, te, tea, test_acc, avg, loss_batch, avg_loss_batch = line.strip().split(' ')
    epoch_list.append(int(epoch))
    train_acc_list_max.append(float(train_acc))
    test_acc_list_max.append(float(test_acc))
    avg_loss_batch_list_max.append(float(avg_loss_batch))
    print("epoch: %s, train_acc: %s, test_acc: %s, avg_loss_batch: %s" % (epoch, train_acc, test_acc, avg_loss_batch))
    # print

In [ ]:
print(train_acc_list_max)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, axs = plt.subplots(3,1,figsize=(6, 16))
# fig.suptitle('[Emotions] Average Precision with various number of layers at mlp_head') #, fontdict={'fontsize': 10}, loc='center', )


#mlp_head with 2 layers
axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list_max, 'o-', markersize=3, label='train accuracy')
# axs[0].plot([i for i in range(1, results_mlp2_mean[:,2].size+1)],    results_mlp2_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# axs[0].plot([i for i in range(1, results_mlp2_meanmax[:,2].size+1)], results_mlp2_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# axs[0].plot([i for i in range(1, results_mlp2_max[:,2].size+1)],     results_mlp2_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# axs[0].plot([i for i in range(1, results_mlp2_maxmax[:,2].size+1)],  results_mlp2_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# axs[0].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# plt.xticks(rotation=45)
# axs[0].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# axs[0].set_yticks(np.arange(0, np.max(results_mlp2_maxmax[:,0])+0.05, 0.001)) #, rotation=45)
# axs[0].set_ylim(0.216, 0.234)
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Train accuracy')
axs[0].grid(True, linestyle='-')
# axs[0].set_title('Train accuracy', fontdict={'fontsize': 10}, loc='center', )
# fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

#mlp_head with 4 layers
axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list_max, 'o-', markersize=3, label='test accuracy')
# axs[1].plot([i for i in range(1, results_mlp4_mean[:,2].size+1)],    results_mlp4_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# axs[1].plot([i for i in range(1, results_mlp4_meanmax[:,2].size+1)], results_mlp4_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# axs[1].plot([i for i in range(1, results_mlp4_max[:,2].size+1)],     results_mlp4_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# axs[1].plot([i for i in range(1, results_mlp4_maxmax[:,2].size+1)],  results_mlp4_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# axs[1].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# plt.xticks(rotation=45)
# axs[1].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# axs[1].set_yticks(np.arange(0, np.max(results_mlp4_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# axs[1].set_ylim(0.212, 0.234)
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Test accuracy')
axs[1].grid(True, linestyle='-')
# axs[1].set_title('Test accuracy', fontdict={'fontsize': 10}, loc='center', )
# fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

#mlp_head with 6 layers
axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list_max, 'o-', markersize=3, label='average loss/batch')
# axs[2].plot([i for i in range(1, results_mlp6_mean[:,2].size+1)],    results_mlp6_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# axs[2].plot([i for i in range(1, results_mlp6_meanmax[:,2].size+1)], results_mlp6_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# axs[2].plot([i for i in range(1, results_mlp6_max[:,2].size+1)],     results_mlp6_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# axs[2].plot([i for i in range(1, results_mlp6_maxmax[:,2].size+1)],  results_mlp6_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
# axs[2].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# plt.xticks(rotation=45)
# axs[2].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# axs[2].set_yticks(np.arange(0, np.max(results_mlp6_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# axs[2].set_ylim(0.216, 0.234)
axs[2].set_xlabel('Epoch')
axs[2].set_ylabel('Average Loss/Batch')
axs[2].grid(True, linestyle='-')
# axs[2].set_title('Average Loss/Batch', fontdict={'fontsize': 10}, loc='center', )


fig.savefig('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/'+current_file_name+ 'VGG19_transfer_train_test_acc_loss.png',dpi=300)

# The original VGG19

In [ ]:
fusion_type = ""
model_file = 'proposed_vgg19_' + str(date.today()) + '_' + data_name + '_' + fusion_type
net = Vgg19(data_len=len(data), no_labels=50, data_name="esc50", fusion_type="") # data_len=len(data), no_labels=3, by CHS
net.train(shuffle_flag=True)
net.save_weights(filepath=base_dir + model_file + '.weights.h5')#, save_traces=False)
tf.keras.models.save_model(net, filepath=base_dir + model_file +'.keras')#, save_traces=False)

In [ ]:
import os
current_file = 'logs_2024-09-11.txt'
current_file_name, ext =current_file.split('.')
# Open and extract the lines of the data file
with open(os.path.join('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/',  data_name, current_file), 'r') as f:
    lines = f.readlines()
f.close()

epoch_list, train_acc_list, test_acc_list, avg_loss_batch_list = [], [], [], []
# loop through lines of the file
for line in lines:
    # remove newline and extract data from line
    e, epoch, tr, tra, train_acc, te, tea, test_acc, avg, loss_batch, avg_loss_batch = line.strip().split(' ')
    epoch_list.append(int(epoch))
    train_acc_list.append(float(train_acc))
    test_acc_list.append(float(test_acc))
    avg_loss_batch_list.append(float(avg_loss_batch))
    print("epoch: %s, train_acc: %s, test_acc: %s, avg_loss_batch: %s" % (epoch, train_acc, test_acc, avg_loss_batch))
    # print

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, axs = plt.subplots(3,1,figsize=(6, 16))
# fig.suptitle('[Emotions] Average Precision with various number of layers at mlp_head') #, fontdict={'fontsize': 10}, loc='center', )


#mlp_head with 2 layers
axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list, 'o-', markersize=3, label='original')
axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list_mean, 'o-', markersize=3, label='mean fusion output')
axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list_mean_max, 'o-', markersize=3, label='mean+max fusion output')
axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list_max, 'o-', markersize=3, label='max fusion output')
axs[0].plot([i for i in range(1, np.size(epoch_list)+1)],        train_acc_list_max_max, 'o-', markersize=3, label='max+max fusion output')
# axs[0].plot([i for i in range(1, results_mlp2_mean[:,2].size+1)],    results_mlp2_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# axs[0].plot([i for i in range(1, results_mlp2_meanmax[:,2].size+1)], results_mlp2_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# axs[0].plot([i for i in range(1, results_mlp2_max[:,2].size+1)],     results_mlp2_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# axs[0].plot([i for i in range(1, results_mlp2_maxmax[:,2].size+1)],  results_mlp2_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
axs[0].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# plt.xticks(rotation=45)
# axs[0].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# axs[0].set_yticks(np.arange(0, np.max(results_mlp2_maxmax[:,0])+0.05, 0.001)) #, rotation=45)
# axs[0].set_ylim(0.216, 0.234)
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Train accuracy')
axs[0].grid(True, linestyle='-')
axs[0].set_title('Train accuracy', fontdict={'fontsize': 10}, loc='center', )
# fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

#mlp_head with 4 layers
axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list, 'o-', markersize=3, label='original')
axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list_mean, 'o-', markersize=3, label='mean fusion output')
axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list_mean_max, 'o-', markersize=3, label='mean+max fusion output')
axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list_max, 'o-', markersize=3, label='max fusion output')
axs[1].plot([i for i in range(1, np.size(epoch_list)+1)],        test_acc_list_max_max, 'o-', markersize=3, label='max+max fusion output')
# axs[1].plot([i for i in range(1, results_mlp4_mean[:,2].size+1)],    results_mlp4_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# axs[1].plot([i for i in range(1, results_mlp4_meanmax[:,2].size+1)], results_mlp4_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# axs[1].plot([i for i in range(1, results_mlp4_max[:,2].size+1)],     results_mlp4_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# axs[1].plot([i for i in range(1, results_mlp4_maxmax[:,2].size+1)],  results_mlp4_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
axs[1].legend(loc=(0.40, 0.01), fontsize=9) #(1.04, 0) :outer loc
# plt.xticks(rotation=45)
# axs[1].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# axs[1].set_yticks(np.arange(0, np.max(results_mlp4_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# axs[1].set_ylim(0.212, 0.234)
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Test accuracy')
axs[1].grid(True, linestyle='-')
axs[1].set_title('Test accuracy', fontdict={'fontsize': 10}, loc='center', )
# fig.savefig('[IurbanEvent]Accuracy_num-layers_mlphead.png',dpi=300)

#mlp_head with 6 layers
axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list, 'o-', markersize=3, label='vgg19 original')
axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list_mean, 'o-', markersize=3, label='mean fusion output')
axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list_mean_max, 'o-', markersize=3, label='mean+max fusion output')
axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list_max, 'o-', markersize=3, label='max fusion output')
axs[2].plot([i for i in range(1, np.size(epoch_list)+1)],        avg_loss_batch_list_max_max, 'o-', markersize=3, label='max+max fusion output')
# axs[2].plot([i for i in range(1, results_mlp6_mean[:,2].size+1)],    results_mlp6_mean[:,2], 'o-', markersize=3, label='SSAST+Fusion mean pooling')
# axs[2].plot([i for i in range(1, results_mlp6_meanmax[:,2].size+1)], results_mlp6_meanmax[:,2], 'o-', markersize=3, label='SSAST+Fusion mean+max pooling')
# axs[2].plot([i for i in range(1, results_mlp6_max[:,2].size+1)],     results_mlp6_max[:,2], 'o-', markersize=3, label='SSAST+Fusion max pooling')
# axs[2].plot([i for i in range(1, results_mlp6_maxmax[:,2].size+1)],  results_mlp6_maxmax[:,2], 'o-', markersize=3, label='SSAST+Fusion max+max pooling')
axs[2].legend(loc=(0.40, 0.60), fontsize=9) #(1.04, 0) :outer loc
# plt.xticks(rotation=45)
# axs[2].set_xticks(np.arange(0, 50+1, 5)) #[Hyosun]results[:,0].size+1 해서 마지막 epoch 보이게
# axs[2].set_yticks(np.arange(0, np.max(results_mlp6_maxmax[:,2])+0.05, 0.001)) #, rotation=45)
# axs[2].set_ylim(0.216, 0.234)
axs[2].set_xlabel('Epoch')
axs[2].set_ylabel('Average Loss/Batch')
axs[2].grid(True, linestyle='-')
axs[2].set_title('Average Loss/Batch', fontdict={'fontsize': 10}, loc='center', )


fig.savefig('/gdrive/MyDrive/ColabNotebooks/Github/TransferLearning/model/esc50/'+'ESC_all_fusion_type'+ str(date.today()) + 'VGG19_transfer_train_test_acc_loss.png',dpi=300)